In [1]:
"""
==============================================================
  이탈 예측 최종 통합 모델 (churn_model_final.py)
  실험 결과 기반 베스트 피처 + 베스트 파라미터 집약본
  Target: ROC-AUC 최대화
==============================================================
"""

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import entropy as scipy_entropy
import warnings
warnings.filterwarnings('ignore')


# ──────────────────────────────────────────────
# 0. 데이터 로드 (경로 수정)
# ──────────────────────────────────────────────
BASE = '/Users/rim/Desktop/workspace/project_1'

train_cust   = pd.read_csv(f'{BASE}/train/train_customer_info.csv')
train_trans  = pd.read_csv(f'{BASE}/train/train_transaction_history.csv')
train_fin    = pd.read_csv(f'{BASE}/train/train_finance_profile.csv')
train_target = pd.read_csv(f'{BASE}/train/train_targets.csv')

test_cust    = pd.read_csv(f'{BASE}/test/test_customer_info.csv')
test_trans   = pd.read_csv(f'{BASE}/test/test_transaction_history.csv')
test_fin     = pd.read_csv(f'{BASE}/test/test_finance_profile.csv')

print("✅ 데이터 로드 완료")


# ──────────────────────────────────────────────
# 1. 피처 엔지니어링 함수
# ──────────────────────────────────────────────
def build_features(cust_df, trans_df, fin_df, target_df=None):
    """
    고객 기본 정보 + 금융 프로필 + 거래 이력을 결합해
    모델 입력용 피처 데이터프레임을 반환합니다.
    target_df가 None이면 테스트 모드로 동작합니다.
    """

    # ── 1-1. 기본 병합 (1:1) ──────────────────
    df = pd.merge(cust_df, fin_df, on='customer_id', how='left')
    if target_df is not None:
        df = pd.merge(df, target_df, on='customer_id', how='left')

    # ── 1-2. 날짜 처리 ────────────────────────
    df['join_date']   = pd.to_datetime(df['join_date'])
    trans_df = trans_df.copy()
    trans_df['trans_date'] = pd.to_datetime(trans_df['trans_date'])
    trans_df['month'] = trans_df['trans_date'].dt.month

    ref_date = trans_df['trans_date'].max()
    df['days_since_joined'] = (ref_date - df['join_date']).dt.days

    # ── 1-3. 범주형 인코딩 ────────────────────
    cat_cols = ['gender', 'region_code', 'prefer_category', 'income_group']
    for col in cat_cols:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))

    # ── 1-4. 기본 파생 변수 ───────────────────
    df['net_asset']          = df['total_deposit_balance'] - df['total_loan_balance']
    df['debt_to_deposit']    = df['total_loan_balance'] / (df['total_deposit_balance'] + 1)
    df['cash_service_ratio'] = df['card_cash_service_amt'] / (df['total_deposit_balance'] + 1)

    # 재정 위기 복합 지표 (표준화 합산)
    distress_vars = ['fin_overdue_days', 'card_cash_service_amt',
                     'card_loan_amt', 'total_loan_balance']
    distress_z = df[distress_vars].copy()
    for v in distress_vars:
        mu, sd = distress_z[v].mean(), distress_z[v].std() + 1e-9
        distress_z[v] = (distress_z[v] - mu) / sd
    df['fin_distress_v2'] = distress_z.sum(axis=1)

    # 고위험 자산 플래그
    df['is_high_risk_asset'] = (
        (df['fin_asset_trend_score'] <= -2) & (df['credit_score'] < 600)
    ).astype(int)

    # ── 1-5. 거래 이력 집계 ───────────────────
    # 전체 집계
    agg = trans_df.groupby('customer_id').agg(
        trans_count      = ('trans_id',       'count'),
        amt_sum          = ('trans_amount',    'sum'),
        amt_mean         = ('trans_amount',    'mean'),
        amt_max          = ('trans_amount',    'max'),
        amt_std          = ('trans_amount',    'std'),
        installment_ratio= ('is_installment',  'mean'),
        online_ratio     = ('biz_type',        lambda x: (x == 'Online').mean()),
        recency          = ('trans_date',
                            lambda x: (ref_date - x.max()).days),
    ).reset_index()
    agg['spending_per_trans'] = agg['amt_sum'] / (agg['trans_count'] + 1)
    agg['amt_std'] = agg['amt_std'].fillna(0)

    df = pd.merge(df, agg, on='customer_id', how='left')
    fill_cols = ['trans_count','amt_sum','amt_mean','amt_max',
                 'amt_std','installment_ratio','online_ratio',
                 'recency','spending_per_trans']
    df[fill_cols] = df[fill_cols].fillna(0)

    # ── 1-6. 월별 Wide 피처 (핵심 시계열) ────
    monthly_amt = trans_df.groupby(['customer_id','month'])['trans_amount'] \
        .sum().unstack(fill_value=0)
    monthly_amt.columns = [f'amt_m{int(c)}' for c in monthly_amt.columns]

    monthly_cnt = trans_df.groupby(['customer_id','month'])['trans_id'] \
        .count().unstack(fill_value=0)
    monthly_cnt.columns = [f'cnt_m{int(c)}' for c in monthly_cnt.columns]

    df = df.merge(monthly_amt.reset_index(), on='customer_id', how='left')
    df = df.merge(monthly_cnt.reset_index(), on='customer_id', how='left')

    # 월별 컬럼 결측치 0 처리
    month_cols = [c for c in df.columns if c.startswith('amt_m') or c.startswith('cnt_m')]
    df[month_cols] = df[month_cols].fillna(0)

    # 후반(10~12월) vs 전반(7~9월) 소비 감소율
    first_half_cols  = [c for c in df.columns if c in ['amt_m7','amt_m8','amt_m9']]
    second_half_cols = [c for c in df.columns if c in ['amt_m10','amt_m11','amt_m12']]
    if first_half_cols and second_half_cols:
        df['amt_first_half']    = df[first_half_cols].sum(axis=1)
        df['amt_second_half']   = df[second_half_cols].sum(axis=1)
        df['half_decline_ratio'] = df['amt_second_half'] / (df['amt_first_half'] + 1)

    # 12월 거래 없는 고객 (강력한 이탈 선행 신호)
    last_month_buyers = trans_df[trans_df['month'] == trans_df['month'].max()]['customer_id'].unique()
    df['no_last_month_trans'] = (~df['customer_id'].isin(last_month_buyers)).astype(int)

    # 최근 소비 감소 플래그
    if 'amt_m12' in df.columns and 'amt_m11' in df.columns:
        df['recent_spending_drop'] = (df['amt_m12'] < df['amt_m11']).astype(int)

    # spending trend ratio (최근 1달 vs 이전 5달 평균)
    max_month = trans_df['month'].max()
    recent_sum = trans_df[trans_df['month'] == max_month] \
        .groupby('customer_id')['trans_amount'].sum()
    older_mean = trans_df[trans_df['month'] < max_month] \
        .groupby('customer_id')['trans_amount'].mean()
    df['spending_trend_ratio'] = (
        df['customer_id'].map(recent_sum).fillna(0) /
        (df['customer_id'].map(older_mean).fillna(1) + 1)
    )

    # ── 1-7. 엔트로피 피처 ───────────────────
    def calc_entropy(series):
        counts = series.value_counts(normalize=True)
        return scipy_entropy(counts)

    cat_ent = trans_df.groupby('customer_id')['item_category'].agg(calc_entropy)
    df['category_entropy'] = df['customer_id'].map(cat_ent).fillna(0)

    # ── 1-8. 피어 그룹 내 상대 순위 ──────────
    df['credit_rank_in_income']  = df.groupby('income_group')['credit_score'] \
                                     .rank(pct=True)
    df['deposit_rank_in_region'] = df.groupby('region_code')['total_deposit_balance'] \
                                     .rank(pct=True)
    df['age_group'] = pd.cut(df['age'], bins=[0,30,40,50,100],
                              labels=[0,1,2,3]).astype(float)
    df['loan_rank_in_age']       = df.groupby('age_group')['card_loan_amt'] \
                                     .rank(pct=True)

    # ── 1-9. 타겟 인코딩 (train only, OOF 방식) ─
    if target_df is not None:
        def target_encode_oof(df_, col, target_col='target_churn', n_splits=5):
            encoded = pd.Series(index=df_.index, dtype=float)
            kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
            for tr_idx, val_idx in kf.split(df_):
                mean_map = df_.iloc[tr_idx].groupby(col)[target_col].mean()
                encoded.iloc[val_idx] = df_.iloc[val_idx][col].map(mean_map)
            return encoded.fillna(df_[target_col].mean())

        for col in ['region_code', 'prefer_category', 'income_group']:
            if col in df.columns:
                df[f'{col}_churn_rate'] = target_encode_oof(df, col)

    return df


# ──────────────────────────────────────────────
# 2. 피처 목록 정의
# ──────────────────────────────────────────────
FEATURES = [
    # 금융 핵심
    'total_deposit_balance', 'card_loan_amt', 'credit_score',
    'fin_distress_v2', 'net_asset', 'fin_asset_trend_score',
    'num_active_cards', 'fin_overdue_days',
    'debt_to_deposit', 'cash_service_ratio',

    # 거래 집계
    'trans_count', 'amt_sum', 'amt_mean', 'amt_std',
    'spending_per_trans', 'installment_ratio', 'online_ratio', 'recency',

    # 시계열 (월별 wide)
    'amt_m7','amt_m8','amt_m9','amt_m10','amt_m11','amt_m12',
    'cnt_m7','cnt_m8','cnt_m9','cnt_m10','cnt_m11','cnt_m12',
    'half_decline_ratio', 'no_last_month_trans',
    'recent_spending_drop', 'spending_trend_ratio',

    # 피어 그룹 순위
    'credit_rank_in_income', 'deposit_rank_in_region', 'loan_rank_in_age',

    # 엔트로피
    'category_entropy',

    # 고객 속성
    'age', 'is_married', 'days_since_joined', 'is_high_risk_asset',
]


# ──────────────────────────────────────────────
# 3. 학습 데이터 구축
# ──────────────────────────────────────────────
print("\n🔧 피처 엔지니어링 실행 중 (train)...")
train_df = build_features(train_cust, train_trans, train_fin, train_target)

# 타겟 인코딩 피처 추가
te_features = [f'{c}_churn_rate' for c in ['region_code','prefer_category','income_group']
               if f'{c}_churn_rate' in train_df.columns]

final_features = [f for f in FEATURES + te_features if f in train_df.columns]

X = train_df[final_features].copy()
y = train_df['target_churn']

print(f"✅ 학습 피처 수: {len(final_features)}개 | 샘플 수: {len(X):,}명")
print(f"   이탈율: {y.mean():.3%}")


# ──────────────────────────────────────────────
# 4. 모델 학습 (Optuna Trial 31 베스트 파라미터)
# ──────────────────────────────────────────────
BEST_PARAMS = {
    'objective':       'binary',
    'metric':          'auc',
    'verbosity':       -1,
    'boosting_type':   'gbdt',
    'random_state':    42,
    'n_estimators':    5000,
    # Optuna 최적값
    'learning_rate':   0.014369,
    'num_leaves':      31,
    'min_child_samples': 22,
    'feature_fraction':  0.7661,
    'bagging_fraction':  0.7553,
    'bagging_freq':      6,
    'reg_alpha':         0.0172,
    'reg_lambda':        0.0525,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
models    = []
fold_aucs = []

print("\n🚀 5-Fold 교차 검증 시작...")
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**BEST_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(200, verbose=False),
                   lgb.log_evaluation(0)]
    )
    models.append(model)

    pred = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = pred
    auc = roc_auc_score(y_val, pred)
    fold_aucs.append(auc)
    print(f"  Fold {fold+1}: AUC = {auc:.4f}")

cv_auc = roc_auc_score(y, oof_preds)
print(f"\n{'='*45}")
print(f"  CV AUC (OOF): {cv_auc:.4f}")
print(f"  Fold 평균:    {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
print(f"{'='*45}")

# 피처 중요도 Top 15
importances = pd.Series(
    np.mean([m.feature_importances_ for m in models], axis=0),
    index=final_features
).sort_values(ascending=False)
print("\n🔥 피처 중요도 Top 15:")
print(importances.head(15).to_string())


# ──────────────────────────────────────────────
# 5. 테스트 예측 및 제출 파일 생성
# ──────────────────────────────────────────────
print("\n🔧 피처 엔지니어링 실행 중 (test)...")
test_df = build_features(test_cust, test_trans, test_fin, target_df=None)

# 타겟 인코딩 — train 전체 통계로 매핑 (test는 OOF 불필요)
for col in ['region_code', 'prefer_category', 'income_group']:
    col_te = f'{col}_churn_rate'
    if col_te in final_features:
        train_mean_map = train_df.groupby(col)['target_churn'].mean()
        test_df[col_te] = test_df[col].map(train_mean_map).fillna(y.mean())

# 없는 컬럼 0으로 패딩
for f in final_features:
    if f not in test_df.columns:
        test_df[f] = 0

X_test = test_df[final_features].copy()

# 5개 fold 모델 평균으로 최종 예측
test_preds = np.mean([m.predict_proba(X_test)[:, 1] for m in models], axis=0)

# 제출 파일 저장
submission = pd.DataFrame({
    'customer_id':  test_df['customer_id'],
    'target_churn': test_preds,
    'target_ltv':   0.0   # LTV는 별도 모델 필요 — 일단 0으로 플레이스홀더
})
submission.to_csv(f'{BASE}/submission_churn.csv', index=False, encoding='utf-8-sig')

print(f"\n✅ 제출 파일 저장 완료: {BASE}/submission_churn.csv")
print(f"   예측값 범위: {test_preds.min():.4f} ~ {test_preds.max():.4f}")
print(f"   예측 이탈율: {test_preds.mean():.3%}")

✅ 데이터 로드 완료

🔧 피처 엔지니어링 실행 중 (train)...
✅ 학습 피처 수: 45개 | 샘플 수: 60,000명
   이탈율: 9.882%

🚀 5-Fold 교차 검증 시작...
  Fold 1: AUC = 0.8022
  Fold 2: AUC = 0.7850
  Fold 3: AUC = 0.8015
  Fold 4: AUC = 0.8065
  Fold 5: AUC = 0.8007

  CV AUC (OOF): 0.7961
  Fold 평균:    0.7992 ± 0.0074

🔥 피처 중요도 Top 15:
total_deposit_balance     595.4
deposit_rank_in_region    463.6
card_loan_amt             429.0
fin_asset_trend_score     424.2
credit_rank_in_income     394.6
loan_rank_in_age          393.0
credit_score              371.6
cash_service_ratio        347.0
fin_distress_v2           296.8
category_entropy          294.6
half_decline_ratio        291.4
days_since_joined         279.6
net_asset                 268.8
amt_m10                   267.0
amt_m8                    262.0

🔧 피처 엔지니어링 실행 중 (test)...

✅ 제출 파일 저장 완료: /Users/rim/Desktop/workspace/project_1/submission_churn.csv
   예측값 범위: 0.0177 ~ 0.9433
   예측 이탈율: 9.844%


In [2]:
# ... (이전 코드 동일)

# ──────────────────────────────────────────────
# 4. 모델 학습 (결과 출력 형식 수정 버전)
# ──────────────────────────────────────────────
from sklearn.metrics import mean_squared_error

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
models = []
fold_scores = []

print("\n🚀 5-Fold 교차 검증 시작...")
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**BEST_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(200, verbose=False),
                   lgb.log_evaluation(0)]
    )
    models.append(model)

    # 예측 (확률값)
    pred = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = pred
    
    # 지표 계산
    auc = roc_auc_score(y_val, pred)
    mse = mean_squared_error(y_val, pred)
    rmse = np.sqrt(mse)
    
    # 임의의 Final Score 산식 (예시: AUC - (RMSE / 10)) 
    # 출력값 형식을 맞추기 위해 예시 산식을 적용했습니다.
    final_score = auc - (rmse / 2) 

    print(f"Fold {fold+1} AUC: {auc:.15f} "
          f"Fold {fold+1} RMSE: {rmse:.15f} "
          f"Fold {fold+1} MSE: {mse:.15f} "
          f"Fold {fold+1} Final Score: {final_score:.15f}")
    
    fold_scores.append(auc)

cv_auc = roc_auc_score(y, oof_preds)
print(f"\n{'='*45}")
print(f"  CV AUC (OOF): {cv_auc:.4f}")
print(f"  Fold 평균 AUC: {np.mean(fold_scores):.4f}")
print(f"{'='*45}")

# ... (이후 피처 중요도 및 테스트 예측 코드 동일)


🚀 5-Fold 교차 검증 시작...
Fold 1 AUC: 0.802181257083555 Fold 1 RMSE: 0.266425656840721 Fold 1 MSE: 0.070982630623009 Fold 1 Final Score: 0.668968428663195
Fold 2 AUC: 0.784964902470129 Fold 2 RMSE: 0.268204068689298 Fold 2 MSE: 0.071933422461494 Fold 2 Final Score: 0.650862868125480
Fold 3 AUC: 0.801480795458763 Fold 3 RMSE: 0.264429445487811 Fold 3 MSE: 0.069922931640991 Fold 3 Final Score: 0.669266072714858
Fold 4 AUC: 0.806544027774876 Fold 4 RMSE: 0.263433693973261 Fold 4 MSE: 0.069397311120398 Fold 4 Final Score: 0.674827180788245
Fold 5 AUC: 0.800737271122220 Fold 5 RMSE: 0.266742536506687 Fold 5 MSE: 0.071151580782021 Fold 5 Final Score: 0.667366002868876

  CV AUC (OOF): 0.7961
  Fold 평균 AUC: 0.7992


In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, mean_squared_error

BASE = '/Users/rim/Desktop/workspace/project_1'
train = pd.read_csv(f'{BASE}/train_p.csv')
test = pd.read_csv(f'{BASE}/test_p.csv')

CH_FEATS = ['fin_distress_v2', 'fin_overdue_days', 'card_loan_amt', 'fin_asset_trend_score', 
            'debt_to_deposit', 'total_deposit_balance', 'recency', 'trans_count', 
            'amt_m12', 'cnt_m12', 'credit_rank_in_income', 'days_since_joined']

# Target Encoding
for c in ['region_code', 'prefer_category', 'income_group']:
    f = f'{c}_churn_rate'
    train[f] = 0.0
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for tr, val in kf.split(train):
        m = train.iloc[tr].groupby(c)['target_churn'].mean()
        train.loc[train.index[val], f] = train.loc[train.index[val], c].map(m)
    train[f].fillna(train['target_churn'].mean(), inplace=True)
    test[f] = test[c].map(train.groupby(c)['target_churn'].mean()).fillna(train['target_churn'].mean())
    CH_FEATS.append(f)

# --- 수정된 학습 로직 ---
X = train[CH_FEATS]
y = train['target_churn']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))
fold_scores = []

print("\n🚀 Churn 모델 5-Fold 교차 검증 시작...")
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.014369, num_leaves=31, random_state=42, verbosity=-1)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)]
    )

    pred = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = pred
    test_preds += model.predict_proba(test[CH_FEATS])[:, 1] / 5
    
    auc = roc_auc_score(y_val, pred)
    mse = mean_squared_error(y_val, pred)
    rmse = np.sqrt(mse)
    final_score = auc - (rmse / 2) # 요청하신 산식 반영

    print(f"Fold {fold+1} AUC: {auc:.15f} RMSE: {rmse:.15f} MSE: {mse:.15f} Final Score: {final_score:.15f}")
    fold_scores.append(auc)

cv_auc = roc_auc_score(y, oof_preds)
print(f"\n{'='*45}")
print(f"   CV AUC (OOF): {cv_auc:.4f}")
print(f"   Fold 평균 AUC: {np.mean(fold_scores):.4f}")
print(f"{'='*45}")

test['target_churn'] = test_preds
test[['customer_id', 'target_churn']].to_csv(f'{BASE}/pred_churn.csv', index=False)


🚀 Churn 모델 5-Fold 교차 검증 시작...
Fold 1 AUC: 0.798202878873888 RMSE: 0.264915189878130 MSE: 0.070180057828166 Final Score: 0.665745283934823
Fold 2 AUC: 0.782390012821429 RMSE: 0.268807362339355 MSE: 0.072257398047841 Final Score: 0.647986331651751
Fold 3 AUC: 0.796948228687377 RMSE: 0.265186135551154 MSE: 0.070323686488555 Final Score: 0.664355160911800
Fold 4 AUC: 0.801344620411178 RMSE: 0.263568635646409 MSE: 0.069468425696510 Final Score: 0.669560302587973
Fold 5 AUC: 0.794693835765329 RMSE: 0.267830778993052 MSE: 0.071733326176025 Final Score: 0.660778446268803

   CV AUC (OOF): 0.7944
   Fold 평균 AUC: 0.7947


In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

BASE = '/Users/rim/Desktop/workspace/project_1'
train = pd.read_csv(f'{BASE}/train_p.csv')
test = pd.read_csv(f'{BASE}/test_p.csv')

LTV_FEATS = ['amt_sum', 'amt_mean', 'amt_max', 'total_deposit_balance', 'net_asset', 
             'active_months', 'amt_first_half', 'amt_second_half', 'half_growth_ratio', 
             'spend_rank_in_region', 'age', 'days_since_joined']

X = train[LTV_FEATS]
y_log = np.log1p(train['target_ltv']) 

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_log = np.zeros(len(X))
test_preds = np.zeros(len(test))
fold_rmse_scores = []

print("\n🚀 LTV 모델 5-Fold 교차 검증 시작...")
for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y_log.iloc[tr_idx], y_log.iloc[val_idx]

    model = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.01, num_leaves=63, random_state=42, verbosity=-1)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)]
    )

    pred_log = model.predict(X_val)
    oof_preds_log[val_idx] = pred_log
    test_preds += np.expm1(model.predict(test[LTV_FEATS])) / 5
    
    # 지표 계산은 원래 단위(expm1)로 수행
    y_val_real = np.expm1(y_val)
    pred_real = np.expm1(pred_log)
    
    mse = mean_squared_error(y_val_real, pred_real)
    rmse = np.sqrt(mse)
    
    print(f"Fold {fold+1} RMSE: {rmse:.15f} MSE: {mse:.15f}")
    fold_rmse_scores.append(rmse)

cv_rmse = np.sqrt(mean_squared_error(np.expm1(y_log), np.expm1(oof_preds_log)))
print(f"\n{'='*45}")
print(f"   CV RMSE (OOF): {cv_rmse:,.2f}")
print(f"   Fold 평균 RMSE: {np.mean(fold_rmse_scores):,.2f}")
print(f"{'='*45}")

test['target_ltv'] = np.maximum(test_preds, 0)
test[['customer_id', 'target_ltv']].to_csv(f'{BASE}/pred_ltv.csv', index=False)


🚀 LTV 모델 5-Fold 교차 검증 시작...
Fold 1 RMSE: 1549276.975215658312663 MSE: 2400259145933.379394531250000
Fold 2 RMSE: 1491498.262819238472730 MSE: 2224567067992.806152343750000
Fold 3 RMSE: 1475297.160492569673806 MSE: 2176501711757.438720703125000
Fold 4 RMSE: 1518711.500778926536441 MSE: 2306484622598.179687500000000
Fold 5 RMSE: 1525109.695605896646157 MSE: 2325959583631.110839843750000

   CV RMSE (OOF): 1,512,201.85
   Fold 평균 RMSE: 1,511,978.72


In [6]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error # RMSE 계산을 위해 추가

BASE = '/Users/rim/Desktop/workspace/project_1'
train = pd.read_csv(f'{BASE}/train_p.csv')
test = pd.read_csv(f'{BASE}/test_p.csv')

# 피처 리스트
LTV_FEATS = ['amt_sum', 'amt_mean', 'amt_max', 'total_deposit_balance', 'net_asset', 
             'active_months', 'amt_first_half', 'amt_second_half', 'half_growth_ratio', 
             'spend_rank_in_region', 'age', 'days_since_joined']

# 로그 변환
y_log = np.log1p(train['target_ltv']) 

kf = KFold(n_splits=5, shuffle=True, random_state=42)
preds = np.zeros(len(test))
rmse_list = [] # 각 폴드의 RMSE를 저장할 리스트

print("🚀 LTV 모델 학습 및 RMSE 계산 시작...")

for i, (tr, val) in enumerate(kf.split(train[LTV_FEATS])):
    # 학습/검증 데이터 분할
    x_tr, y_tr = train[LTV_FEATS].iloc[tr], y_log.iloc[tr]
    x_val, y_val = train[LTV_FEATS].iloc[val], y_log.iloc[val]
    
    # 모델 학습
    m = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.01, num_leaves=63, random_state=42, verbosity=-1)
    m.fit(x_tr, y_tr)
    
    # 검증셋 예측 (로그 상태에서 예측)
    val_pred_log = m.predict(x_val)
    
    # 실제값과 예측값 모두 지수 복원 (원래 단위인 원/달러 등으로 복구)
    val_pred_real = np.expm1(val_pred_log)
    val_y_real = np.expm1(y_val)
    
    # RMSE 계산 (squared=False 옵션이 RMSE를 바로 반환합니다)
    rmse = mean_squared_error(val_y_real, val_pred_real, squared=False)
    rmse_list.append(rmse)
    
    print(f"  - Fold {i+1} RMSE: {rmse:,.2f}")
    
    # 테스트셋 예측 및 지수 복원 후 누적
    preds += np.expm1(m.predict(test[LTV_FEATS])) / 5

# 결과 출력
print("\n" + "="*30)
print(f"✅ 평균 RMSE (LTV): {np.mean(rmse_list):,.2f}")
print("="*30)

#test['target_ltv'] = np.maximum(preds, 0)
#test[['customer_id', 'target_ltv']].to_csv(f'{BASE}/pred_ltv.csv', index=False)
#print("✅ pred_ltv.csv 생성 완료")

🚀 LTV 모델 학습 및 RMSE 계산 시작...


TypeError: got an unexpected keyword argument 'squared'

In [7]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error # 기본 함수 유지

# ... (데이터 로드 및 피처 설정 부분 동일) ...

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y_log.iloc[tr_idx], y_log.iloc[val_idx]

    model = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.01, num_leaves=63, random_state=42, verbosity=-1)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)]
    )

    pred_log = model.predict(X_val)
    oof_preds_log[val_idx] = pred_log
    
    # 실제값 복원
    y_val_real = np.expm1(y_val)
    pred_real = np.expm1(pred_log)
    
    # MSE 계산 후 루트 씌워서 RMSE 도출 (TypeError 해결)
    mse = mean_squared_error(y_val_real, pred_real)
    rmse = np.sqrt(mse)
    
    print(f"Fold {fold+1} RMSE: {rmse:.15f} MSE: {mse:.15f}")
    fold_rmse_scores.append(rmse)

# 최종 CV RMSE도 동일하게 수정
cv_mse = mean_squared_error(np.expm1(y_log), np.expm1(oof_preds_log))
cv_rmse = np.sqrt(cv_mse)

print(f"\n✅ 최종 CV RMSE (OOF): {cv_rmse:,.2f}")

Fold 1 RMSE: 1549276.975215658312663 MSE: 2400259145933.379394531250000
Fold 2 RMSE: 1491498.262819238472730 MSE: 2224567067992.806152343750000
Fold 3 RMSE: 1475297.160492569673806 MSE: 2176501711757.438720703125000
Fold 4 RMSE: 1518711.500778926536441 MSE: 2306484622598.179687500000000
Fold 5 RMSE: 1525109.695605896646157 MSE: 2325959583631.110839843750000

✅ 최종 CV RMSE (OOF): 1,512,201.85
